In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                # self.p.v_pd_hf_tweezer_squeeze_power = .984
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 450.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.28 * np.pi

                self.p.t_tweezer_hold = 15.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 20

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=False)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [9]:
eBuilder = ExptBuilder()

In [ ]:
img_amp = np.linspace(.2,1.9,50)

freq_raman = np.array([1.19497570e+08, 1.19504282e+08, 1.19510994e+08, 1.19517706e+08,
       1.19524418e+08, 1.19531130e+08, 1.19537842e+08, 1.19544554e+08,
       1.19551266e+08, 1.19557978e+08, 1.19564690e+08, 1.19571401e+08,
       1.19578113e+08, 1.19584825e+08, 1.19591537e+08, 1.19598249e+08,
       1.19604961e+08, 1.19611673e+08, 1.19618385e+08, 1.19625097e+08,
       1.19631809e+08, 1.19638520e+08, 1.19645232e+08, 1.19651944e+08,
       1.19658656e+08, 1.19665368e+08, 1.19672080e+08, 1.19678792e+08,
       1.19685504e+08, 1.19692216e+08, 1.19698928e+08, 1.19705640e+08,
       1.19712351e+08, 1.19719063e+08, 1.19725775e+08, 1.19732487e+08,
       1.19739199e+08, 1.19745911e+08, 1.19752623e+08, 1.19759335e+08,
       1.19766047e+08, 1.19772759e+08, 1.19779471e+08, 1.19786182e+08,
       1.19792894e+08, 1.19799606e+08, 1.19806318e+08, 1.19813030e+08,
       1.19819742e+08, 1.19826454e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119496519.0
0 Failed to connect to Monitor: [WaxxClient] Server 'monitor' not discovered within 3.0 s. Is the server running and on the 192.168.1.x subnet?
 10 values of dummy. 10 total shots. 30 total images expected.
Run ID: 68813
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.28, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.28 pi, x-center = 994, y-center = 820

 Run ID: 68813

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.28, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.28 pi, x-center = 994, y-center = 820

Wavemeter exposure time 581 ms is too long for reliable lock detection.
Wavemeter exposure time 581 ms is too long for reliable lock detection.
Error getting frequency from wavemeter
laser ry_405 unlocked:
target = 741.091200, meas = 0.000000, diff = -741091200.0
shot 1/10 don

: 